<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/03_select_hyperparam_for_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [ ]:
do = False # @param{type:"boolean"}
if do :
    %pip install torchinfo -qq
    %pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
    %pip install starfile -qq
    %pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [ ]:
if do :
    %pip install pycuda==2024.1
    %pip install "numpy<2.0"
    %pip install mrcfile -qq

## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [ ]:
# @title  { display-mode: "form" }

INPUT_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
IMAGE_DIR = INPUT_IMAGE_DIR
# @markdown ---

use_denoised_as_pariwise = False # @param {type : "boolean"}
dnzd_pw = use_denoised_as_pariwise
DENOISED_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
# @markdown ---
LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_diff_post_processing_but_circularity_MAD/10017/unet_eb5_dice_CRF" # @param {type:"string"}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

In [ ]:
IMAGE_DIR = "/content/image_dir"

In [ ]:
!git clone https://github.com/phonchi/CryoParticleSegment.git

!wget -O /content/CryoParticleSegment/Modeling/convcrf.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/convcrf.py
!wget -O /content/CryoParticleSegment/Modeling/dataset.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/dataset.py
!wget -O /content/CryoParticleSegment/Modeling/model.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/model.py
!wget -O /content/CryoParticleSegment/Modeling/trainer.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/trainer.py

Cloning into 'CryoParticleSegment'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 270 (delta 25), reused 8 (delta 8), pack-reused 233 (from 3)
Receiving objects: 100% (270/270), 32.08 MiB | 20.58 MiB/s, done.
Resolving deltas: 100% (114/114), done.
--2026-02-09 16:19:38--  https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/convcrf.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17753 (17K) [text/plain]
Saving to: ‘/content/CryoParticleSegment/Modeling/convcrf.py’

/content/CryoPartic 100%[===================>]  17.34K  --.-KB/s    in 0.001s  

2026-02-09 16:19:38 (28.9 MB/s) - ‘/content/CryoParticleSegm

In [ ]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

### ✅ Packages Handling

In [ ]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

In [ ]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery, collate_fn

## ⭐ Main

### ✅ Setting

In [ ]:
# @markdown Parameters.

user = True # @param {type:"boolean"}

In [ ]:
# @markdown Parameters.

BATCH = 8
CROP_SIZE = (512, 512)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

### ✅ Dataset

You can provide a [`transforms.CenterCrop(3840)`](https://docs.pytorch.org/vision/master/generated/torchvision.transforms.CenterCrop.html) object to crop out boundary artifacts.


In [ ]:
crop = transforms.CenterCrop(3840)

In [ ]:
train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
if dnzd_pw == False:
    train_dataset = MicrographDataset(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE, num_patches = 4, crop=crop)
else:
    dnzd_train_dir = os.path.join(DENOISED_IMAGE_DIR, 'train')
    train_dataset = MicrographDataset(image_dir=train_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_train_dir, filenames=train_filenames, crop_size=CROP_SIZE, num_patches = 4, crop=crop)

In [ ]:
val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
if dnzd_pw == False:
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE, crop=crop)
else:
    dnzd_val_dir = os.path.join(DENOISED_IMAGE_DIR, 'val')
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_val_dir, filenames=val_filenames, crop_size=CROP_SIZE, crop=crop)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)

In [ ]:
if not user:
    test_dir = os.path.join(IMAGE_DIR, 'test')
    test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
    if dnzd_pw == False:
        test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=LABEL_DIR, filenames=test_filenames, crop_size=CROP_SIZE)
    else:
        dnzd_test_dir = os.path.join(DENOISED_IMAGE_DIR, 'test')
        test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_test_dir, filenames=test_filenames, crop_size=CROP_SIZE)

    test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)

In [ ]:
for i1, i2, i3, i4, i5 in val_loader: #test loader and reconstruct
    print(i3.dtype, i5.dtype)
    print(i3.shape, i5.shape)
    shape = i5.shape
    break

torch.int64 torch.int64
torch.Size([81, 1, 512, 512]) torch.Size([1, 3840, 3840])


## ⭐ Evaluate

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
import starfile
import pandas as pd
import matplotlib
from PIL import Image
import cv2

def get_basename_with_uid_removed(path):
  return os.path.basename(path).split(sep='_', maxsplit=1)[-1]


def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy


In [ ]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}
SIZE = 4096 # @param {type:"integer"}

In [ ]:
!cp {DATASET_DIR}/{EMPIAR_ID}/filtered_val.star .

In [ ]:
y_size = SIZE
labeled_particles = starfile.read(f"filtered_val.star")['particles']
labeled_particles = labeled_particles[['rlnMicrographName', 'rlnCoordinateX', 'rlnCoordinateY']]
labeled_particles.columns = pd.Index(['image_name', 'x_coord', 'y_coord'])
labeled_particles['image_name'] = labeled_particles['image_name'].apply(get_basename_with_uid_removed)
labeled_particles['image_name'] = labeled_particles['image_name'].apply(lambda s: s.split(".")[0])
labeled_particles['y_coord'] = y_size - labeled_particles['y_coord']
labeled_particles

,image_name,x_coord,y_coord
0,Falcon_2012_06_13-03_22_02_0,2169,1426
1,Falcon_2012_06_13-03_22_02_0,2791,1957
2,Falcon_2012_06_13-03_22_02_0,2372,475
3,Falcon_2012_06_13-03_22_02_0,2635,1047
4,Falcon_2012_06_13-03_22_02_0,3560,3965
...,...,...,...
3540,Falcon_2012_06_12-15_14_01_0,1810,1593
3541,Falcon_2012_06_12-15_14_01_0,1178,1780
3542,Falcon_2012_06_12-15_14_01_0,364,1047
3543,Falcon_2012_06_12-15_14_01_0,961,556


In [ ]:
def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

def plot_micrograph_and_labels(ax, micrograph, labels, coords):
    ax.imshow(micrograph, cmap='gray')
    ax.imshow(labels, cmap='gray', alpha=0.5)
    for x, y in coords:
        corrected_x, corrected_y = x, y
        circle = matplotlib.patches.Circle((corrected_x, corrected_y), radius=RADIUS, fill=False, color='r')
        ax.add_patch(circle)

You can specify a `crop_size` in `preprocess_and_crop()` to remove boundary artifacts during preprocessing.

In [ ]:
label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)
gts = []

for idx, (test_image, _, _, grid, _) in enumerate(val_dataset):
    # if idx == 6:
    #     break
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)
    print(cropped_label_image.shape)
    label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    coords = locations[['x_coord', 'y_coord']].values - BORDER
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, coords)
    plt.show()
    print(len(coords))
    gts.append(coords)
    ##

#filename = f"{os.path.splitext(checkpoint_path)[0]}.png"
#pred_path = os.path.join(RESULT_DIR, "Each_ckpt", filename)
#save_image(pred_image, pred_path)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
label_images.shape

(6, 3840, 3840)

### CV approach

In [ ]:
def get_mad_range(data, z_thresh=3.0):
    arr = np.array(data)
    if len(arr) == 0: return 0.0, 1.0
    med = np.median(arr)
    mad = np.median(np.abs(arr - med))
    low = med - (z_thresh * mad / 0.6745)
    high = med + (z_thresh * mad / 0.6745)
    return max(0.0, low), min(1.0, high)

In [ ]:
radius = RADIUS

In [ ]:
from center_finding import normalize, min_rect_circle, eliminate_near

In [ ]:
cv_list_all = []
cv_config = []
e_factor = [2,4,6]
s_factor = [0.6,1,1.4]

for e in e_factor:
    for s in s_factor:
        cv_list = []
        print(f"e_factor: {e}, s_factor: {s}")
        for img in label_images:
            thresh1 = normalize(img)
            kernel_size = int(radius / 4)  # Example ratio
            kernel = np.ones((kernel_size, kernel_size), np.uint8)
            thresh1 = cv2.erode(thresh1, kernel, iterations=1)

            kernel_size = int(radius / e)  # Example ratio
            kernel = np.ones((kernel_size, kernel_size), np.uint8)
            thresh1 = cv2.erode(thresh1, kernel, iterations=1)

            contours, hierarchy = cv2.findContours(thresh1,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

            # 1. Initial Area Filtering & Circularity Calculation (changed)
            contr_min = radius**s
            candidates = []
            circ_values = []

            for cnt in contours:
                area = cv2.contourArea(cnt)
                if contr_min < area < 500000:
                    perimeter = cv2.arcLength(cnt, True)
                    if perimeter > 0:
                        circularity = (4 * np.pi * area) / (perimeter ** 2)
                        candidates.append((cnt, circularity))
                        circ_values.append(circularity)

            # 2. Apply MAD Filtering (changed)
            if circ_values:
                c_low, c_high = get_mad_range(circ_values, z_thresh=3.0)
                # Filter candidates that fall within the statistical MAD range
                c_full_list = [c[0] for c in candidates if c_low <= c[1] <= c_high]
            else:
                c_full_list = []

            # 3. Final Mapping to custom function
            c_list = (list(map(lambda x: min_rect_circle(x, radius), c_full_list)))
            c_list = [x for x in c_list if x is not None]

            cv_list.append(c_list)
            # print(f"Total Contours: {len(contours)}, Validated: {len(c_list)}")

        cv_list_all.append(cv_list)
        cv_config.append((e,s))

e_factor: 2, s_factor: 0.6
e_factor: 2, s_factor: 1
e_factor: 2, s_factor: 1.4
e_factor: 4, s_factor: 0.6
e_factor: 4, s_factor: 1
e_factor: 4, s_factor: 1.4
e_factor: 6, s_factor: 0.6
e_factor: 6, s_factor: 1
e_factor: 6, s_factor: 1.4


In [ ]:
observe_id = 0 # @param {type:"integer"}

In [ ]:
for idx, (test_image, _, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    #locations = labeled_particles[labeled_particles['image_name'] == name]
    #adjusted_c_list = [(x + 128, y + 128) for x, y in cv_list_all[0][idx]]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, cv_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### TrackPy

More information about the Crocker–Grier centroid-finding algorithm is available in the [TrackPy documentation](https://soft-matter.github.io/trackpy/dev/generated/trackpy.locate.html).


In [ ]:
import trackpy as tp
from skimage.measure import regionprops, label

In [ ]:
TrackParticleSize = RADIUS*2-1
curr_adpmass = 0
sep = [0.4, 0.7, 1]
#topn = 500
scale = [0.15, 0.25, 0.35]  # Scale factor (0.5 means reducing the size to half)

In [ ]:
label_images.shape

(6, 3840, 3840)

In [ ]:
tp_list_all = []
tp_config = []
for e in scale:
    for s in sep:
        print(f"e_factor: {e}, s_factor: {s}")
        tp_list = []
        for img in label_images:
            output_image = img
            small_image = cv2.resize(output_image, None, fx=e, fy=e, interpolation=cv2.INTER_AREA)
            # Adjust parameters based on the scale
            sep_s = round(s * TrackParticleSize)+1
            small_sep = int(sep_s * e)
            small_diameter = int(TrackParticleSize * e)
            # Ensure small_diameter is odd
            if small_diameter % 2 == 0:
                small_diameter += 1

            """
            coorTrack = tp.locate(small_image, diameter=small_diameter, minmass=curr_adpmass, separation=small_sep)
            #coorTrack.loc[:,'prob']=[output_image[getattr(coor,'y'),getattr(coor,'x')] for coor in np.floor(coorTrack[['x','y']]).astype('int').itertuples()]
            """

            # 1. Coordinate Detection
            coorTrack = tp.locate(small_image, diameter=small_diameter, minmass=curr_adpmass, separation=small_sep)

            if len(coorTrack) > 0:
                # 2. Shape Analysis for MAD Filtering (changed)
                # Create a local mask to calculate circularity at detection points
                mask = (small_image > small_image.mean()).astype(np.uint8)
                labeled_mask = label(mask)
                props = regionprops(labeled_mask)

                circularities = []
                # Map centroids to region properties
                for idx, row in coorTrack.iterrows():
                    py, px = int(row['y']), int(row['x'])
                    # Ensure indices are within mask bounds
                    py = np.clip(py, 0, labeled_mask.shape[0]-1)
                    px = np.clip(px, 0, labeled_mask.shape[1]-1)

                    region_idx = labeled_mask[py, px]
                    if region_idx > 0:
                        region = props[region_idx - 1]
                        area = region.area
                        perim = region.perimeter
                        circ = (4 * np.pi * area) / (perim ** 2) if perim > 0 else 0
                        circularities.append(circ)
                    else:
                        circularities.append(0)

                coorTrack['circularity'] = circularities

                # 3. Calculate and Apply MAD Range (changed)
                valid_circs = [c for c in circularities if c > 0]
                c_low, c_high = get_mad_range(valid_circs, z_thresh=3.0)

                coorTrack = coorTrack[(coorTrack['circularity'] >= c_low) &
                                      (coorTrack['circularity'] <= c_high)]

            coorTrack['x'] *= (1/e)
            coorTrack['y'] *= (1/e)
            coords = coorTrack[['x', 'y']].values
            tp_list.append(coords)
            # print(len(coords))

        tp_list_all.append(tp_list)
        tp_config.append((e,s))

e_factor: 0.15, s_factor: 0.4
e_factor: 0.15, s_factor: 0.7
e_factor: 0.15, s_factor: 1
e_factor: 0.25, s_factor: 0.4
e_factor: 0.25, s_factor: 0.7
e_factor: 0.25, s_factor: 1
e_factor: 0.35, s_factor: 0.4
e_factor: 0.35, s_factor: 0.7
e_factor: 0.35, s_factor: 1


In [ ]:
observe_id = 3 # @param {type:"integer"}

In [ ]:
for idx, (test_image, _, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, tp_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### Nonmax

In [ ]:
import pycuda.driver as drv
from center_finding import cleanCanList, pad_image, reLev, reNorm, scoreGpu, getMax, getMax3, gaussian_kernel_2d_opencv, reshape

In [ ]:
#ratio=1
pnum=2000 #initial filtering, larger if candidate is more 2000 for 10017, should inverse proportional to pCans. Use 1000 for 10081
pCans=[0.4, 0.5, 0.6] #the grid size smaller will generate more candidate
overlaps=[0, 0.3, 0.6] #allow for overlap, smaller will delete more
psize=RADIUS*2

# Nor affect too much
level=3
nSep=10
nIter=100

In [ ]:
nms_list_all = []
nms_config = []
for e in pCans:
    for s in overlaps:
        pCan=e
        overlap=s
        print(f"e_factor: {e}, s_factor: {s}")
        nms_list = []
        for img in label_images:
            heatArr=normalize(pad_image(img))
            heatArr=reNorm(heatArr, nSep)
            heatArr=reLev(heatArr,level)
            gaus= gaussian_kernel_2d_opencv(kernel_size = psize,sigma = 0)

            # canList,score=scoreGpuLaunch(heatArr,gaus,psize, gsize, nIter, fscore)
            gsize=psize*pCan
            heat=heatArr.astype(np.float32)
            gaus=gaus.astype(np.float32)
            score=np.zeros(heat.shape).astype(np.float32)
            [sizex,sizey]=heat.shape
            sizex = int(sizex)
            sizey = int(sizey)
            psize = int(psize)
            gsize = int(gsize)

            func = scoreGpu

            tx=16
            ty=16
            bx=(sizex-1)//tx+1
            by=(sizey-1)//ty+1
            print('get score gpu',tx, ty, bx, by, gsize, psize)


            func(drv.In(heat), drv.In(gaus), drv.Out(score), np.int32(sizex), np.int32(sizey), np.int32(psize),
                block=(tx, ty, 1), grid=(int(bx), int(by)))

            # # Calculate necessary dimensions and convert all to int
            numx = int((sizex - 1) // gsize + 1)
            numy = int((sizey - 1) // gsize + 1)
            num = numx * numy
            tnum = num * 5

            # # Create the result array
            res = np.zeros(tnum).astype(np.float32)

            # # Block and grid dimensions
            bx = (numx - 1) // tx + 1
            by = (numy - 1) // ty + 1

            # Get the function from the module
            func = getMax

            # Call the function, ensure all parameters are the correct type
            func(drv.In(score), drv.Out(res), np.int32(gsize), np.int32(sizex), np.int32(sizey), np.int32(numx),
                block=(16, 16, 1), grid=(bx, by, 1))

            # Make sure all dimensions and sizes are properly cast to np.int32 to avoid ambiguity
            niter = np.int32(nIter)
            gsize = np.int32(gsize)
            sizex = np.int32(sizex)
            sizey = np.int32(sizey)
            numx = np.int32(numx)
            tnum = np.int32(num)

            print('get Max3', tx, ty, bx, by, niter, 'other : ', gsize, sizex, sizey, numx, tnum)
            func = getMax3
            # Ensuring the correct parameter order and type for the kernel invocation
            func(drv.In(score), drv.InOut(res), gsize, sizex, sizey, numx, tnum, niter,
                    block=(16, 16, 1), grid=(bx, by, 1))
            """
            canList=reshape(res,num)

            print('Number of Particles before:', len(canList))
            if(len(canList)>pnum):
                canList=canList[:pnum]


            canList=cleanCanList(canList,overlap,psize)
            #canList=reCan(canList,ratio)
            print('Number of Particles:', len(canList))
            nms_list.append([(r[1],r[0]) for r in canList])
        nms_list_all.append(nms_list)
        nms_config.append((e,s))
        print("\n")
        """

            canList = reshape(res, num)
            if len(canList) > pnum:
                canList = canList[:pnum]

            canList = cleanCanList(canList, overlap, psize)

            # --- MAD Circularity Filtering (changed) ---
            if len(canList) > 0:
                # Create a binary mask from the processed heatmap
                mask = (heatArr > heatArr.mean()).astype(np.uint8)
                labeled_mask = label(mask)
                props = regionprops(labeled_mask)

                circularities = []
                temp_candidates = []

                for r in canList:
                    # Assuming r[0] is y and r[1] is x based on your append logic
                    py, px = int(r[0]), int(r[1])

                    if 0 <= py < labeled_mask.shape[0] and 0 <= px < labeled_mask.shape[1]:
                        region_idx = labeled_mask[py, px]
                        if region_idx > 0:
                            region = props[region_idx - 1]
                            area = region.area
                            perim = region.perimeter
                            circ = (4 * np.pi * area) / (perim ** 2) if perim > 0 else 0
                            circularities.append(circ)
                            temp_candidates.append({'data': r, 'circ': circ})

                if circularities:
                    c_low, c_high = get_mad_range(circularities, z_thresh=3.0)
                    # Keep only statistically 'normal' circular shapes
                    canList = [c['data'] for c in temp_candidates if c_low <= c['circ'] <= c_high]
            # --- End MAD Filtering ---

            print(f"Final Particles: {len(canList)}")
            nms_list.append([(r[1], r[0]) for r in canList])

        nms_list_all.append(nms_list)
        nms_config.append((e, s))
        print("\n")

e_factor: 0.4, s_factor: 0
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Final Particles: 194
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Final Particles: 198
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Final Particles: 241
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Final Particles: 179
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Final Particles: 199
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Final Particles: 180


e_factor: 0.4, s_factor: 0.3
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Final Particles: 338
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Final Particles: 335
get sc

In [ ]:
observe_id = 2 # @param {type:"integer"}

In [ ]:
for idx, (test_image, _, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, nms_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### Score

> #### 🗒 Info
> Here, we compute the score based on the validation set. You may choose to rank the algorithms using either the F-score or mAP. Additionally, the parameter $\beta$ in the F-score can be adjusted: values of $\beta > 1$ place greater emphasis on recall over precision, while $\beta < 1$ give more weight to precision.


In [ ]:
from metrics import centers_to_boxes, calculate_iou_torchvision, evaluate_detection_raw_multiple, f_beta_score, calculate_mAP_multiple_images

In [ ]:
# Assign a default confidence score of 1.0 to all predicted boxes
default_score = 1.0
beta = 1 # @param {type:"number"}
F_score = False # @param {type:"boolean"}

In [ ]:
width = RADIUS*2
height = width
cv_scores = []
for i, cv_list in enumerate(cv_list_all):
    print(f"config {cv_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(cv_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    cv_scores.append((f_beta, mAP_value.item()))

config (2, 0.6)
Precision: 0.9836063981056213
Recall: 0.8057481646537781
F_Beta: 0.8858379065947974
Mean Average Precision (mAP): 0.6729276776313782
config (2, 1)
Precision: 0.9965359568595886
Recall: 0.6492450833320618
F_Beta: 0.7862480543333482
Mean Average Precision (mAP): 0.5398146510124207
config (2, 1.4)
Precision: 0.9981684684753418
Recall: 0.2082686573266983
F_Beta: 0.3446299890298928
Mean Average Precision (mAP): 0.16960926353931427
config (4, 0.6)
Precision: 0.9879746437072754
Recall: 0.8340393900871277
F_Beta: 0.9045043056481143
Mean Average Precision (mAP): 0.7278139591217041
config (4, 1)
Precision: 0.9879746437072754
Recall: 0.8340393900871277
F_Beta: 0.9045043056481143
Mean Average Precision (mAP): 0.7278139591217041
config (4, 1.4)
Precision: 0.9879746437072754
Recall: 0.8340393900871277
F_Beta: 0.9045043056481143
Mean Average Precision (mAP): 0.7278139591217041
config (6, 0.6)
Precision: 0.9931291937828064
Recall: 0.7626941800117493
F_Beta: 0.8627904918032318
Mean Aver

In [ ]:
width = RADIUS*2
height = width
tp_scores = []

for i, tp_list in enumerate(tp_list_all):
    print(f"config {tp_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(tp_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    tp_scores.append((f_beta, mAP_value.item()))

config (0.15, 0.4)
Precision: 1.0
Recall: 0.8110808730125427
F_Beta: 0.8956870839935435
Mean Average Precision (mAP): 0.6442918181419373
config (0.15, 0.7)
Precision: 1.0
Recall: 0.6667925715446472
F_Beta: 0.8000906446645827
Mean Average Precision (mAP): 0.5386597514152527
config (0.15, 1)
Precision: 1.0
Recall: 0.36542654037475586
F_Beta: 0.5352562434804594
Mean Average Precision (mAP): 0.30400967597961426
config (0.25, 0.4)
Precision: 0.9997184872627258
Recall: 0.7152204513549805
F_Beta: 0.8338712144052108
Mean Average Precision (mAP): 0.6165072321891785
config (0.25, 0.7)
Precision: 1.0
Recall: 0.5624699592590332
F_Beta: 0.7199753901518492
Mean Average Precision (mAP): 0.49901628494262695
config (0.25, 1)
Precision: 1.0
Recall: 0.34778085350990295
F_Beta: 0.5160792314332244
Mean Average Precision (mAP): 0.313805490732193
config (0.35, 0.4)
Precision: 0.9992344379425049
Recall: 0.6586883664131165
F_Beta: 0.7939864242930069
Mean Average Precision (mAP): 0.5633614659309387
config (0.35

In [ ]:
width = RADIUS*2
height = width
nms_scores = []

for i, nms_list in enumerate(nms_list_all):
    print(f"config {nms_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(nms_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    nms_scores.append((f_beta, mAP_value.item()))

config (0.4, 0)
Precision: 0.9923746585845947
Recall: 0.34473273158073425
F_Beta: 0.5117076299504497
Mean Average Precision (mAP): 0.2017037272453308
config (0.4, 0.3)
Precision: 1.0
Recall: 0.5604022741317749
F_Beta: 0.7182792327620632
Mean Average Precision (mAP): 0.4930155277252197
config (0.4, 0.6)
Precision: 0.9988057017326355
Recall: 0.7060349583625793
F_Beta: 0.8272817026732849
Mean Average Precision (mAP): 0.6155080795288086
config (0.5, 0)
Precision: 0.9864203333854675
Recall: 0.3606926202774048
F_Beta: 0.5282326679084943
Mean Average Precision (mAP): 0.1935359388589859
config (0.5, 0.3)
Precision: 0.9968433976173401
Recall: 0.6105620265007019
F_Beta: 0.7572883801695971
Mean Average Precision (mAP): 0.49515852332115173
config (0.5, 0.6)
Precision: 0.977351188659668
Recall: 0.7387753129005432
F_Beta: 0.8414798437752938
Mean Average Precision (mAP): 0.5997217297554016
config (0.6, 0)
Precision: 0.9670570492744446
Recall: 0.3568049669265747
F_Beta: 0.5212790370293177
Mean Average

## watershed

In [ ]:
# @title class function define
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
import math

class WatershedDetector:
    def __init__(self, particle_radius=64, border=0, margin=40, z_thresh=3.0, use_contour_area=True):
        self.radius = particle_radius
        self.border = border
        self.margin = margin
        self.z_thresh = z_thresh
        self.use_contour_area = use_contour_area
        self.expected_area = np.pi * (self.radius ** 2)

    def _get_contour_expected_area(self, mask):
        """Calculates expected area based on actual mask contours."""
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours: return self.expected_area
        areas = [cv2.contourArea(cnt) for cnt in contours]
        median_area = np.median(areas)
        return median_area if median_area > 0 else self.expected_area

    def _calculate_metrics(self, region):
        """Generates the full dictionary of shape descriptors."""
        area = region.area
        perimeter = region.perimeter
        if perimeter == 0: return None
        return {
            'area': area,
            'circularity': (4 * np.pi * area) / (perimeter ** 2),
            'eccentricity': region.eccentricity,
            'solidity': area / region.convex_area if region.convex_area > 0 else 0,
            'hu_0': region.moments_hu[0],
            'hu_1': region.moments_hu[1],
            'hu_2': region.moments_hu[2]
        }

    def _get_mad_range(self, data):
        """Calculates the filtering window using Median Absolute Deviation."""
        arr = np.array(data)
        med = np.median(arr)
        mad = np.median(np.abs(arr - med))
        if mad == 0: return np.min(arr), np.max(arr)
        low = med - (self.z_thresh * mad / 0.6745)
        high = med + (self.z_thresh * mad / 0.6745)
        return max(0.0, low), min(1.0, high)

    def _plot_all_histograms(self, candidates, c_bounds):
        """Generates an auto-adjusting grid of histograms for all metrics. (changed)"""
        # Convert candidate metrics list of dicts to a DataFrame for easy plotting
        metrics_df = pd.DataFrame([c['metrics'] for c in candidates])
        metric_names = metrics_df.columns.tolist()

        n_metrics = len(metric_names)
        n_cols = 3
        n_rows = math.ceil(n_metrics / n_cols)

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
        axes = axes.flatten() # Flatten to 1D for easy iteration

        for idx, col_name in enumerate(metric_names):
            ax = axes[idx]
            ax.hist(metrics_df[col_name], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
            ax.set_title(f"Distribution: {col_name}")
            ax.set_xlabel("Value")
            ax.set_ylabel("Frequency")

            # Only add bounds for Circularity as requested (changed)
            if col_name == 'circularity':
                ax.axvline(c_bounds[0], color='red', linestyle='--', linewidth=2, label=f'Lower: {c_bounds[0]:.2f}')
                ax.axvline(c_bounds[1], color='blue', linestyle='--', linewidth=2, label=f'Upper: {c_bounds[1]:.2f}')
                ax.legend(loc='upper right', fontsize='small')

        # Hide any unused axes in the grid
        for i in range(idx + 1, len(axes)):
            axes[i].axis('off')

        plt.tight_layout()
        plt.show()

    def detect(self, prob_map_input, peak_thresh=0.4, dist_ratio=0.8, show_plots=True):
        """Main detection entry point with optional visualization. (changed)"""
        prob_map = prob_map_input.astype(np.float32)
        if prob_map.max() > 1.0: prob_map /= 255.0
        mask = (prob_map > 0.5).astype(np.uint8)
        if np.sum(mask) == 0: mask = (prob_map > 0.05).astype(np.uint8)

        current_expected_area = self._get_contour_expected_area(mask) if self.use_contour_area else self.expected_area

        distance = ndi.distance_transform_edt(mask)
        coords = peak_local_max(distance, min_distance=int(self.radius * dist_ratio),
                                labels=mask, threshold_abs=self.radius * peak_thresh)
        markers = np.zeros(distance.shape, dtype=int)
        for i, (r, c) in enumerate(coords): markers[r, c] = i + 1
        labels = watershed(-distance, markers, mask=mask)

        candidates, circularities = [], []
        for region in regionprops(labels):
            if current_expected_area * 0.3 <= region.area <= current_expected_area * 2.0:
                m = self._calculate_metrics(region)
                if m:
                    candidates.append({'region': region, 'metrics': m})
                    circularities.append(m['circularity'])

        if not candidates: return []

        c_bounds = self._get_mad_range(circularities)

        # Trigger histogram plotting if enabled (changed)
        if show_plots:
            self._plot_all_histograms(candidates, c_bounds)

        final_particles = []
        for cand in candidates:
            if c_bounds[0] <= cand['metrics']['circularity'] <= c_bounds[1]:
                ry, rx = cand['region'].centroid
                if (self.margin < rx < prob_map.shape[1] - self.margin and
                    self.margin < ry < prob_map.shape[0] - self.margin):

                    final_particles.append({
                        'x': int(rx),
                        'y': int(ry),
                        'score': float(prob_map[int(ry), int(rx)]),
                        **cand['metrics']
                    })
        return final_particles

In [ ]:
# @title DT-Only Grid Search
import itertools

watershed_list_all = []
watershed_config = []

# Search Space
threshold_levels_ratios = [0.3, 0.35, 0.4]
min_dist_ratios = [0.75, 0.8, 0.85]

# Initialize detector
detector = WatershedDetector(particle_radius=RADIUS, border=BORDER)

print(f"Starting DT-Only Grid Search using WatershedDetector class...")

param_grid = list(itertools.product(threshold_levels_ratios, min_dist_ratios))

for thresh_r, dist_r in param_grid:
    print(f"Testing Config: Thresh={thresh_r}, Dist={dist_r}")
    watershed_list = []

    for img in label_images:
        particles = detector.detect(
            img,
            peak_thresh=thresh_r,
            dist_ratio=dist_r,
            show_plots=False
        )

        # Extract (x, y) coordinates from the list of dictionaries returned by detector
        coords = [(p['x'], p['y']) for p in particles] if particles else []
        watershed_list.append(coords)

    watershed_list_all.append(watershed_list)
    watershed_config.append((thresh_r, dist_r))

print(f" -> Finished generating {len(watershed_list_all)} configurations.")

Starting DT-Only Grid Search using WatershedDetector class...
Testing Config: Thresh=0.3, Dist=0.75
Testing Config: Thresh=0.3, Dist=0.8
Testing Config: Thresh=0.3, Dist=0.85
Testing Config: Thresh=0.35, Dist=0.75
Testing Config: Thresh=0.35, Dist=0.8
Testing Config: Thresh=0.35, Dist=0.85
Testing Config: Thresh=0.4, Dist=0.75
Testing Config: Thresh=0.4, Dist=0.8
Testing Config: Thresh=0.4, Dist=0.85
 -> Finished generating 9 configurations.


In [ ]:
# @title Visualize DT-Only Results
import os

selected_indices = range(len(val_filenames))
print(f"Visualizing {len(selected_indices)} images...")

for i, ws_list in enumerate(watershed_list_all):
    cfg = watershed_config[i]
    print(f"\n{'='*20} Param Val: Thresh={cfg[0]}, Dist={cfg[1]} {'='*20}")

    for idx in selected_indices:
        name = val_filenames[idx][:-4]

        # 1. Load Background (Micrograph)
        micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
        if os.path.exists(micrograph_path):
            micrograph = np.load(micrograph_path)
            bg_image = preprocess_and_crop(micrograph)
        else:
            print(f"Warning: {name}.npy not found.")
            continue

        # 2. Load Label (Ground Truth)
        label_path = f"{LABEL_DIR}/{name}.png"
        if os.path.exists(label_path):
            image = Image.open(label_path)
            lbl_raw = np.array(image)
            current_label_image = preprocess_and_crop(lbl_raw)
        else:
            current_label_image = np.zeros_like(bg_image)

        # 3. Get Predictions using coords from ws_list (Changed)
        # ws_list[idx] now contains [(x, y), ...] from the grid search
        coords = ws_list[idx]

        # 4. Plot
        _, ax = plt.subplots(figsize=(12, 12))
        plot_micrograph_and_labels(ax, bg_image, current_label_image, coords)

        plt.title(f"Img {idx} ({name}) | Particles: {len(coords)} (Thresh={cfg[0]}, Dist={cfg[1]})")
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# @title DT-Only Grid Search: Evaluate & Score
import torch
import os
import json

# Metrics Helper
def evaluate_detection_robust(iou_matrices, iou_threshold=0.5):
    tp_total = 0; fp_total = 0; fn_total = 0
    for iou_matrix in iou_matrices:
        n_gt, n_pred = iou_matrix.shape
        if n_gt == 0 and n_pred == 0: continue
        if n_gt > 0 and n_pred == 0: fn_total += n_gt; continue
        if n_gt == 0 and n_pred > 0: fp_total += n_pred; continue

        max_vals, _ = torch.max(iou_matrix, dim=1)
        detected_mask = max_vals >= iou_threshold
        tp = torch.sum(detected_mask).item()
        tp_total += tp
        fn_total += (n_gt - tp)
        fp_total += (n_pred - tp)

    epsilon = 1e-6
    precision = tp_total / (tp_total + fp_total + epsilon)
    recall = tp_total / (tp_total + fn_total + epsilon)
    return precision, recall

# Main Eval Loop
width = RADIUS * 2
watershed_scores = []

use_MAP = False # @param{type:"boolean"}
print("Evaluating DT-Only Results using Class-derived detections...")

for i, ws_list in enumerate(watershed_list_all):
    cfg = watershed_config[i] # (thresh, dist)

    iou_matrices = []
    gt_full = []
    pred_full = []
    scores_list = []

    for idx, gt in enumerate(gts):
        gt_boxes = centers_to_boxes(np.array(gt), width, width)
        gt_full.append(gt_boxes)

        # Handle class-method output structure (Changed)
        preds = ws_list[idx] # This is [(x, y), ...] or list of dicts depending on your grid search loop

        if len(preds) > 0:
            # If your grid search stored tuples, we use a dummy score of 1.0 for mAP
            # If it stored dicts, we use p['distance_val']
            coords = np.array([p[:2] for p in preds])

            # Using 1.0 as default score if not provided by detector (Changed)
            sc = torch.ones(len(preds), dtype=torch.float32)
            pred_boxes = centers_to_boxes(coords, width, width)
        else:
            pred_boxes = torch.empty((0, 4))
            sc = torch.tensor([], dtype=torch.float32)

        pred_full.append(pred_boxes)
        scores_list.append(sc)

        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    # 1. F-Beta
    precision, recall = evaluate_detection_robust(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=1)

    # 2. mAP
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    try:
        mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores_list, iou_thresholds)
        map_val = mAP_value.item()
    except:
        map_val = 0.0

    print(f"Cfg: {cfg} | F1: {f_beta:.4f}, mAP: {map_val:.4f}")
    watershed_scores.append((f_beta, map_val))

# Save Best
if len(watershed_scores) > 0:
    best_idx = max(range(len(watershed_scores)), key=lambda i: watershed_scores[i][use_MAP])
    best_cfg = watershed_config[best_idx]
    method = "DT_defalut" # @param{type:"string"}
    best_res = {
        "method": method,
        "radius": RADIUS,
        "peak_thresh": best_cfg[0],
        "min_dist": best_cfg[1],
        "f_score": watershed_scores[best_idx][0],
        "map": watershed_scores[best_idx][1]
    }

    print("\n--- BEST CONFIGURATION (DT ONLY - NO AR) ---")
    print(best_res)

    # 1. Save to local content first (Current Directory)
    local_file = 'best_watershed_params.json' # @param{type:"string"}
    with open(local_file, 'w') as f:
        json.dump(best_res, f, indent=4)
    print(f"✅ Saved locally to: {local_file}")

    # 2. Copy to RESULT_DIR using shell command
    # We put quotes "" around the path in case RESULT_DIR has spaces
    !cp {local_file} "{RESULT_DIR}"

    print(f"✅ Copied to: {RESULT_DIR}/{local_file}")

Evaluating DT-Only Results using Class-derived detections...
Cfg: (0.3, 0.75) | F1: 0.9402, mAP: 0.8642
Cfg: (0.3, 0.8) | F1: 0.9303, mAP: 0.8469
Cfg: (0.3, 0.85) | F1: 0.9150, mAP: 0.8233
Cfg: (0.35, 0.75) | F1: 0.9402, mAP: 0.8642
Cfg: (0.35, 0.8) | F1: 0.9303, mAP: 0.8469
Cfg: (0.35, 0.85) | F1: 0.9150, mAP: 0.8233
Cfg: (0.4, 0.75) | F1: 0.9401, mAP: 0.8640
Cfg: (0.4, 0.8) | F1: 0.9295, mAP: 0.8465
Cfg: (0.4, 0.85) | F1: 0.9141, mAP: 0.8218

--- BEST CONFIGURATION (DT ONLY - NO AR) ---
{'method': 'DT_defalut', 'radius': 64, 'peak_thresh': 0.3, 'min_dist': 0.75, 'f_score': 0.9402426236512678, 'map': 0.8642399907112122}
✅ Saved locally to: best_watershed_params.json
✅ Copied to: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_diff_post_processing_but_circularity_MAD/10017/unet_eb5_dice_CRF/best_watershed_params.json


## Ending

### Write the best hyperparameters

In [ ]:
if F_score:
    cv_scores_sorted = sorted(cv_scores, key=lambda x: x[0], reverse=True)
    csorted_indices = sorted(range(len(cv_scores)), key=lambda i: cv_scores[i][0], reverse=True)
else:
    cv_scores_sorted = sorted(cv_scores, key=lambda x: x[1], reverse=True)
    csorted_indices = sorted(range(len(cv_scores)), key=lambda i: cv_scores[i][1], reverse=True)
cv_scores_sorted[0], cv_config[csorted_indices[0]]

((0.9045043056481143, 0.7278139591217041), (4, 0.6))

In [ ]:
if F_score:
    tp_scores_sorted = sorted(tp_scores, key=lambda x: x[0], reverse=True)
    tsorted_indices = sorted(range(len(tp_scores)), key=lambda i: tp_scores[i][0], reverse=True)
else:
    tp_scores_sorted = sorted(tp_scores, key=lambda x: x[1], reverse=True)
    tsorted_indices = sorted(range(len(tp_scores)), key=lambda i: tp_scores[i][1], reverse=True)
tp_scores_sorted[0], tp_config[tsorted_indices[0]]

((0.8956870839935435, 0.6442918181419373), (0.15, 0.4))

In [ ]:
if F_score:
    nms_scores_sorted = sorted(nms_scores, key=lambda x: x[0], reverse=True)
    nsorted_indices = sorted(range(len(nms_scores)), key=lambda i: nms_scores[i][0], reverse=True)
else:
    nms_scores_sorted = sorted(nms_scores, key=lambda x: x[1], reverse=True)
    nsorted_indices = sorted(range(len(nms_scores)), key=lambda i: nms_scores[i][1], reverse=True)
nms_scores_sorted[0], nms_config[nsorted_indices[0]]

((0.8272817026732849, 0.6155080795288086), (0.4, 0.6))

In [ ]:
with open("best_config.txt", "w") as f:
    f.write(f"cv_config: {cv_config[csorted_indices[0]]}\n")
    f.write(f"tp_config: {tp_config[tsorted_indices[0]]}\n")
    f.write(f"nms_config: {nms_config[nsorted_indices[0]]}\n")

In [ ]:
!cp best_config.txt {RESULT_DIR}